In [5]:
# Standard imports
import polars as pl
import numpy as np
import pandas as pd
from pathlib import Path

import joblib

# Preprocessing
from sklearn.preprocessing import OrdinalEncoder

# LightGBM
import lightgbm as lgb

# Metrics
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.calibration import calibration_curve

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

import time

# Paths
TRAIN_PATH = Path("../data/interim/train_split.parquet")
VAL_PATH = Path("../data/interim/val_split.parquet")

print(f"LightGBM version: {lgb.__version__}")
print(f"Polars version: {pl.__version__}")
print(f"Train file exists: {TRAIN_PATH.exists()}")

LightGBM version: 4.3.0
Polars version: 1.39.3
Train file exists: True


## Load and preprocess data for LightGBM

In [6]:
# Add time features and is_app flag
def add_features(df):
    """
    Extract hour_of_day, day_of_week, and is_weekend from 'hour' column and is_app flag. 
    Drop columns that are not useful anymore or problematic.
    """
    return (
        df
        .with_columns(
            pl.col("hour").cast(pl.String).alias("hour_str")
        )
        .with_columns([
            pl.col("hour_str").str.slice(4, 2).cast(pl.Int8).alias("day"),
            pl.col("hour_str").str.slice(6, 2).cast(pl.Int8).alias("hour_of_day"),
        ])
        .with_columns(
            ((pl.col("day") - 21 + 1) % 7).alias("day_of_week")
        )
        .with_columns(pl.col("day_of_week").is_in([5, 6]).alias("is_weekend"))
        .with_columns((pl.col("app_id") != "ecad2386").alias("is_app"))
        .drop("hour_str", "id","hour", "day","device_id", "device_ip")
    )

# Load data and apply features
print("Loading and preparing train...")
train = add_features(pl.scan_parquet(TRAIN_PATH).collect())
print(f"Train shape: {train.shape}")

# Load data and apply features
print("Loading and preparing train...")
val = add_features(pl.scan_parquet(VAL_PATH).collect())
print(f"Val shape: {val.shape}")

Loading and preparing train...
Train shape: (32377421, 24)
Loading and preparing train...
Val shape: (8051546, 24)


In [7]:
# Separate features and target, convert to pandas
y_train = train["click"].to_numpy()
X_train = train.drop("click").to_pandas()

y_val = val["click"].to_numpy()
X_val = val.drop("click").to_pandas()

print(f"X_train type: {type(X_train).__name__}")
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}")

X_train type: DataFrame
X_train shape: (32377421, 23)
y_train shape: (32377421,)
X_val shape: (8051546, 23)


In [8]:
CATEGORICAL_COLS = [
    # Integer columns
    "C1", "banner_pos", "device_type", "device_conn_type",
    "C14", "C15", "C16", "C17", "C18", "C19", "C20", "C21",
    "hour_of_day", "day_of_week",
    # Object/string columns
    "site_id", "site_domain", "site_category",
    "app_id", "app_domain", "app_category", "device_model",
]

# Ordinal encoding
encoder = OrdinalEncoder(
    handle_unknown = "use_encoded_value",
    unknown_value = -1,
    dtype = np.int32
)

print("Encoding categorical columns...")
X_train[CATEGORICAL_COLS] = encoder.fit_transform(X_train[CATEGORICAL_COLS])
X_val[CATEGORICAL_COLS] = encoder.transform(X_val[CATEGORICAL_COLS])      

for col in ["is_app", "is_weekend"]:
    for X_set in [X_train,X_val]:
        X_set[col] = X_set[col].astype("int8")

# Get indices of categorical columns
categorical_indices = [X_train.columns.get_loc(col) for col in CATEGORICAL_COLS]

print(f"X_train shape: {X_train.shape}")
print("Ready for training.")

Encoding categorical columns...
X_train shape: (32377421, 23)
Ready for training.


In [ ]:
joblib.dump(ordinal_encoder, "../models/ordinal_encoder.pkl")

In [9]:
print("Creating LightGBM Dataset with indices for train...")
start = time.time()

train_data = lgb.Dataset(
    X_train,
    label = y_train,
    categorical_feature = categorical_indices,
    free_raw_data=False
)

print(f"Train dataset created in {time.time() - start:.1f} s")

print("\nCreating LightGBM Dataset with indices for val")
start = time.time()

val_data = lgb.Dataset(
    X_val,
    label = y_val,
    categorical_feature = categorical_indices,
    reference = train_data,
    free_raw_data=False
)

print(f"Val dataset created in {time.time() - start:.1f} s")

Creating LightGBM Dataset with indices for train...
Train dataset created in 0.0 s

Creating LightGBM Dataset with indices for val
Val dataset created in 0.0 s


### Training parameters

In [10]:
params_first = {
    "objective": "binary",
    "metric": "binary_logloss",
    "num_leaves": 63,
    "min_data_in_leaf": 100,
    "learning_rate": 0.1,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "lambda_l1": 0.1,
    "lambda_l2": 0.1,
    "n_jobs": 1,
    "verbose": 1,  # show LightGBM internal logs
}
print("Parameters set.")

Parameters set.


## Model training

In [11]:
print("Training LightGBM...")
start = time.time()

model_first = lgb.train(
    params = params_first,
    train_set = train_data,
    num_boost_round = 500,
    valid_sets = [train_data, val_data],
    valid_names = ["train", "val"],
    callbacks = [
    lgb.early_stopping(stopping_rounds = 20),
    lgb.log_evaluation(period = 25)
    ]
)

elapsed = time.time() - start

print(f"\n Training completed in: {elapsed:.1f} s ({elapsed / 60:.2f} min)")


Training LightGBM...
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 5550840, number of negative: 26826581
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 2.877467 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6016
[LightGBM] [Info] Number of data points in the train set: 32377421, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.171442 -> initscore=-1.575444
[LightGBM] [Info] Start training from score -1.575444
Training until validation scores don't improve for 20 rounds
[25]	train's binary_logloss: 0.400606	val's binary_logloss: 0.396051
[50]	train's binary_logloss: 0.395934	val's binary_l

NameError: name 'model' is not defined

In [34]:
print(f"Best iteration: {model_first.best_iteration}")

# Saving model
model_first.save_model("../models/lightgbm_default.txt")
print("Model_first saved to ../models/lightgbm_default.txt")

Best iteration: 211
Model_first saved to ../models/lightgbm_default.txt


In [28]:
y_val_pred_first = model_first.predict(X_val.values, num_iteration=model_first.best_iteration)

log_loss_first = log_loss(y_val, y_val_pred_first)
auc_first = roc_auc_score(y_val, y_val_pred_first)

print("LightGBM baseline model:")
print(f"  Log loss: {log_loss_first:.4f}")
print(f"  AUC:      {auc_first:.4f}")

LightGBM baseline model:
  Log loss: 0.3889
  AUC:      0.7466


## Hyperparameter tuning

### Experiment 1. - lower learning rate, more trees

In [13]:
params_exp1 = {
    "objective": "binary",
    "metric": "binary_logloss",
    "num_leaves": 63,
    "min_data_in_leaf": 100,
    "learning_rate": 0.05, # change 0.1 --> 0.05
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "lambda_l1": 0.1,
    "lambda_l2": 0.1,
    "n_jobs": 1,
    "verbose": 1,
}

print("Experiment 1 parameters:")
print(f"  learning_rate: {params_exp1['learning_rate']} (was 0.1)")

Experiment 1 parameters:
  learning_rate: 0.05 (was 0.1)


In [14]:
print("Training Experiment 1 - lower learning rate...")
start = time.time()
print(f"num_boost_round: 1000 (was 500)\n")

model_exp1 = lgb.train(
    params=params_exp1,
    train_set=train_data,
    num_boost_round=1000, # increased from 500
    valid_sets=[train_data, val_data],
    valid_names=["train", "val"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=20),
        lgb.log_evaluation(period=50), # less frequent logs (since more iterations)
    ],
)

elapsed = time.time() - start
print(f"\nTraining completed in {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f"Best iteration: {model_exp1.best_iteration}")

Training Experiment 1 - lower learning rate...
num_boost_round: 1000 (was 500)

[LightGBM] [Info] Number of positive: 5550840, number of negative: 26826581
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 2.789196 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6016
[LightGBM] [Info] Number of data points in the train set: 32377421, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.171442 -> initscore=-1.575444
[LightGBM] [Info] Start training from score -1.575444
Training until validation scores don't improve for 20 rounds
[50]	train's binary_logloss: 0.400746	val's binary_logloss: 0.39584
[100]	train's binary_logloss: 0.395922	val's binary_logloss: 0.391797
[150]	train's binary_logloss: 0.393879	val's binary_logloss: 0.390362
[200]	train's binary_logloss: 0.392638	val's binary_logloss: 0.389626
[250]	train's b

In [17]:
model_exp1.save_model("../models/lightgbm_exp1.txt")
print("Model_exp1 saved to ../models/lightgbm_exp1.txt")

Model_exp1 saved to ../models/lightgbm_exp1.txt


In [29]:
# Predict with experiment 1 model
y_val_pred_exp1 = model_exp1.predict(X_val.values, num_iteration=model_exp1.best_iteration)

log_loss_exp1 = log_loss(y_val, y_val_pred_exp1)
auc_exp1 = roc_auc_score(y_val, y_val_pred_exp1)

print("EXPERIMENT 1 (learning_rate=0.05, 617 trees):")
print(f"  Log loss: {log_loss_exp1:.4f}")
print(f"  AUC:      {auc_exp1:.4f}")

EXPERIMENT 1 (learning_rate=0.05, 617 trees):
  Log loss: 0.3886
  AUC:      0.7472


### Experiment 2. - larger trees, higher min_data_in_leaf

In [15]:
params_exp2 = {
    "objective": "binary",
    "metric": "binary_logloss",
    "num_leaves": 127, # changed from 63
    "min_data_in_leaf": 200, # changed from 100
    "learning_rate": 0.1,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "lambda_l1": 0.1,
    "lambda_l2": 0.1,
    "n_jobs": 1,
    "verbose": 1,
}

print("Experiment 2 parameters:")
print(f"  num_leaves: {params_exp2['num_leaves']} (was 63)")
print(f"  min_data_in_leaf: {params_exp2['min_data_in_leaf']} (was 100)")

Experiment 2 parameters:
  num_leaves: 127 (was 63)
  min_data_in_leaf: 200 (was 100)


In [16]:
print("Training Experiment 2 - larger trees...")
start = time.time()

model_exp2 = lgb.train(
    params=params_exp2,
    train_set=train_data,
    num_boost_round=1000,
    valid_sets=[train_data, val_data],
    valid_names=["train", "val"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=20),
        lgb.log_evaluation(period=50),
    ],
)

elapsed = time.time() - start
print(f"\nTraining completed in {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f"Best iteration: {model_exp2.best_iteration}")

Training Experiment 2 - larger trees...
[LightGBM] [Info] Number of positive: 5550840, number of negative: 26826581
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 2.805681 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6016
[LightGBM] [Info] Number of data points in the train set: 32377421, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.171442 -> initscore=-1.575444
[LightGBM] [Info] Start training from score -1.575444
Training until validation scores don't improve for 20 rounds
[50]	train's binary_logloss: 0.393163	val's binary_logloss: 0.390824
[100]	train's binary_logloss: 0.390028	val's binary_logloss: 0.389413
[150]	train's binary_logloss: 0.388588	val's binary_logloss: 0.389067
Early stopping, best iteration is:
[155]	train's binary_logloss: 0.388496	val's binary_logloss: 0.389046

Training completed

In [18]:
model_exp2.save_model("../models/lightgbm_exp2.txt")
print("Model_exp2 saved to ../models/lightgbm_exp2.txt")

Model_exp2 saved to ../models/lightgbm_exp2.txt


In [26]:
y_val_pred_exp2 = model_exp2.predict(X_val.values, num_iteration=model_exp2.best_iteration)

log_loss_exp2 = log_loss(y_val, y_val_pred_exp2)
auc_exp2 = roc_auc_score(y_val, y_val_pred_exp2)

print("EXPERIMENT 2 (aggressive tuning):")
print(f"  Log loss: {log_loss_exp2:.4f}")
print(f"  AUC:      {auc_exp2:.4f}")

EXPERIMENT 2 (aggressive tuning):
  Log loss: 0.3890
  AUC:      0.7458


### Experiment 3. - combination of lower LR + larger trees + stronger regularization

In [19]:
params_exp3 = {
    "objective": "binary",
    "metric": "binary_logloss",
    "num_leaves": 127, # large trees
    "min_data_in_leaf": 300, # more conservative regularization
    "learning_rate": 0.03, # even lower LR
    "feature_fraction": 0.8, # slightly more aggressive feature subsampling
    "bagging_fraction": 0.7, # slightly more aggressive row subsampling
    "bagging_freq": 5,
    "lambda_l1": 0.5, # stronger L1 (was 0.1)
    "lambda_l2": 0.5, # stronger L2 (was 0.1)
    "n_jobs": 1,
    "verbose": 1,
}

print("Experiment 3 parameters:")
print(f"  num_leaves: {params_exp3['num_leaves']} (was 63)")
print(f"  min_data_in_leaf: {params_exp3['min_data_in_leaf']} (was 100)")
print(f"  learning_rate: {params_exp3['learning_rate']} (was 0.1)")
print(f"  feature_fraction: {params_exp3['feature_fraction']} (was 0.9)")
print(f"  bagging_fraction: {params_exp3['bagging_fraction']} (was 0.8)")
print(f"  lambda_l1: {params_exp3['lambda_l1']} (was 0.1)")
print(f"  lambda_l2: {params_exp3['lambda_l2']} (was 0.1)")

Experiment 3 parameters:
  num_leaves: 127 (was 63)
  min_data_in_leaf: 300 (was 100)
  learning_rate: 0.03 (was 0.1)
  feature_fraction: 0.8 (was 0.9)
  bagging_fraction: 0.7 (was 0.8)
  lambda_l1: 0.5 (was 0.1)
  lambda_l2: 0.5 (was 0.1)


In [20]:
print("Training Experiment 3 - aggressive tuning...")
start = time.time()

model_exp3 = lgb.train(
    params=params_exp3,
    train_set=train_data,
    num_boost_round=2000, # more iterations possible due to low LR
    valid_sets=[train_data, val_data],
    valid_names=["train", "val"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=30), # more patience for low LR
        lgb.log_evaluation(period=50),
    ],
)

elapsed = time.time() - start
print(f"\nTraining completed in {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f"Best iteration: {model_exp3.best_iteration}")

model_exp3.save_model("../models/lightgbm_exp3.txt")
print("Model_exp3 saved to ../models/lightgbm_exp3.txt")

Training Experiment 3 - aggressive tuning...
[LightGBM] [Info] Number of positive: 5550840, number of negative: 26826581
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 2.750028 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6016
[LightGBM] [Info] Number of data points in the train set: 32377421, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.171442 -> initscore=-1.575444
[LightGBM] [Info] Start training from score -1.575444
Training until validation scores don't improve for 30 rounds
[50]	train's binary_logloss: 0.403295	val's binary_logloss: 0.398785
[100]	train's binary_logloss: 0.396043	val's binary_logloss: 0.39287
[150]	train's binary_logloss: 0.393684	val's binary_logloss: 0.391165
[200]	train's binary_logloss: 0.392053	val's binary_logloss: 0.389971
[250]	train's binary_logloss: 0.391058	val's binar

In [24]:
y_val_pred_exp3 = model_exp3.predict(X_val.values, num_iteration=model_exp3.best_iteration)

log_loss_exp3 = log_loss(y_val, y_val_pred_exp3)
auc_exp3 = roc_auc_score(y_val, y_val_pred_exp3)

print("EXPERIMENT 3 (aggressive tuning):")
print(f"  Log loss: {log_loss_exp3:.4f}")
print(f"  AUC:      {auc_exp3:.4f}")

EXPERIMENT 3 (aggressive tuning):
  Log loss: 0.3886
  AUC:      0.7469


## Summary

In [31]:
print("\nFULL COMPARISON:")
print(f"  Baseline:    log_loss={log_loss_first:.4f}, AUC={auc_first:.4f}")
print(f"  Exp 1:       log_loss={log_loss_exp1:.4f}, AUC={auc_exp1:.4f}")
print(f"  Exp 2:       log_loss={log_loss_exp2:.4f}, AUC={auc_exp2:.4f}")
print(f"  Exp 3:       log_loss={log_loss_exp3:.4f}, AUC={auc_exp3:.4f}")

print("\nDIFFERENCE vs Default:")
print(f"  Exp 1:       diff_log_loss={log_loss_exp1-log_loss_first:+.4f}, diff_AUC={auc_exp1-auc_first:+.4f}")
print(f"  Exp 2:       diff_log_loss={log_loss_exp2-log_loss_first:+.4f}, diff_AUC={auc_exp2-auc_first:+.4f}")
print(f"  Exp 3:       diff_log_loss={log_loss_exp3-log_loss_first:+.4f}, diff_AUC={auc_exp3-auc_first:+.4f}")


FULL COMPARISON:
  Baseline:    log_loss=0.3889, AUC=0.7466
  Exp 1:       log_loss=0.3886, AUC=0.7472
  Exp 2:       log_loss=0.3890, AUC=0.7458
  Exp 3:       log_loss=0.3886, AUC=0.7469

DIFFERENCE vs Default:
  Exp 1:       diff_log_loss=-0.0003, diff_AUC=+0.0006
  Exp 2:       diff_log_loss=+0.0001, diff_AUC=-0.0008
  Exp 3:       diff_log_loss=-0.0004, diff_AUC=+0.0003


Experiment 1. had the highest AUC increase (0.0006) and lower training time in comparison to experiment 3. (with the lowest log_loss). Therefore, the model for exp1 will be used as the final model.

In [33]:
# Final model = Experiment 1
model_exp1.save_model("../models/lightgbm_final.txt")
print("Final LightGBM model saved.")

# Also save the ordinal encoder (needed for predictions on new data)
import joblib
joblib.dump(encoder, "../models/ordinal_encoder.pkl")
print("OrdinalEncoder saved.")

Final LightGBM model saved.
OrdinalEncoder saved.


In [35]:
from pathlib import Path
models_dir = Path("../models")
for f in models_dir.iterdir():
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name}: {size_mb:.2f} MB")

  .gitkeep: 0.00 MB
  lightgbm_default.txt: 6.90 MB
  lightgbm_exp1.txt: 20.01 MB
  lightgbm_exp2.txt: 10.38 MB
  lightgbm_exp3.txt: 46.80 MB
  lightgbm_final.txt: 20.01 MB
  onehot_encoder.pkl: 0.31 MB
  ordinal_encoder.pkl: 0.31 MB
  sgdclassifier_reg_log.pkl: 0.39 MB
